# 🌱 Module 1: Crop Recommendation System
## Bharat Krishi AI — Intelligent Agriculture Decision Support System

**Objective:** Recommend the most suitable crop based on soil nutrients and environmental conditions.

**Input Features:**
- Nitrogen (N), Phosphorus (P), Potassium (K)
- Temperature, Humidity, pH, Rainfall

**Output:** Recommended Crop with Confidence Score

**Models Used:**
- Decision Tree
- Random Forest
- XGBoost
- Neural Network (MLP)


---
## 📦 Step 1: Install & Import Libraries

In [ ]:
# Install required libraries (run once)
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn plotly

In [ ]:
# Core Libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score
)
from xgboost import XGBClassifier

# Model Saving
import pickle
import joblib

print('✅ All libraries imported successfully!')

---
## 📂 Step 2: Load Dataset
> **Note:** Replace `'Crop_recommendation.csv'` with your actual dataset file path.

In [ ]:
# -------------------------------------------------------
# OPTION A: Load from your CSV file
# df = pd.read_csv('Crop_recommendation.csv')
# -------------------------------------------------------

# OPTION B: Generate sample dataset (if you don't have one yet)
np.random.seed(42)

crop_conditions = {
    'rice':       {'N':(60,100), 'P':(40,60),  'K':(40,60),  'temp':(20,27), 'humidity':(80,90), 'ph':(5.5,7.0), 'rainfall':(200,300)},
    'maize':      {'N':(60,100), 'P':(50,80),  'K':(30,50),  'temp':(18,27), 'humidity':(55,75), 'ph':(5.5,7.5), 'rainfall':(60,110)},
    'chickpea':   {'N':(30,60),  'P':(60,90),  'K':(70,100), 'temp':(18,28), 'humidity':(14,25), 'ph':(6.0,8.0), 'rainfall':(60,100)},
    'kidneybeans':{'N':(15,35),  'P':(60,90),  'K':(15,30),  'temp':(15,27), 'humidity':(15,25), 'ph':(5.5,7.0), 'rainfall':(60,110)},
    'pigeonpeas': {'N':(15,35),  'P':(60,90),  'K':(15,25),  'temp':(18,35), 'humidity':(30,50), 'ph':(5.5,7.0), 'rainfall':(60,100)},
    'mothbeans':  {'N':(15,35),  'P':(40,70),  'K':(15,25),  'temp':(24,35), 'humidity':(40,60), 'ph':(4.5,7.0), 'rainfall':(30,60)},
    'mungbean':   {'N':(15,35),  'P':(40,70),  'K':(15,25),  'temp':(25,35), 'humidity':(80,90), 'ph':(6.0,7.5), 'rainfall':(45,100)},
    'blackgram':  {'N':(30,50),  'P':(50,70),  'K':(15,25),  'temp':(25,35), 'humidity':(60,70), 'ph':(6.0,7.0), 'rainfall':(60,100)},
    'lentil':     {'N':(15,30),  'P':(60,90),  'K':(15,25),  'temp':(15,25), 'humidity':(60,70), 'ph':(6.5,8.0), 'rainfall':(35,60)},
    'pomegranate':{'N':(15,25),  'P':(15,25),  'K':(15,25),  'temp':(18,35), 'humidity':(85,95), 'ph':(5.5,7.5), 'rainfall':(100,200)},
    'banana':     {'N':(80,120), 'P':(60,80),  'K':(40,60),  'temp':(25,35), 'humidity':(75,85), 'ph':(5.5,6.5), 'rainfall':(100,200)},
    'mango':      {'N':(15,25),  'P':(15,25),  'K':(15,25),  'temp':(24,35), 'humidity':(45,55), 'ph':(5.5,7.5), 'rainfall':(90,150)},
    'grapes':     {'N':(15,25),  'P':(15,25),  'K':(15,25),  'temp':(15,25), 'humidity':(75,85), 'ph':(5.5,6.5), 'rainfall':(50,100)},
    'watermelon': {'N':(80,120), 'P':(15,25),  'K':(50,70),  'temp':(24,35), 'humidity':(80,90), 'ph':(5.5,6.5), 'rainfall':(40,80)},
    'muskmelon':  {'N':(80,120), 'P':(15,25),  'K':(50,70),  'temp':(28,38), 'humidity':(90,95), 'ph':(5.5,6.5), 'rainfall':(20,40)},
    'apple':      {'N':(15,25),  'P':(120,145),'K':(195,210),'temp':(21,24), 'humidity':(90,95), 'ph':(5.5,7.0), 'rainfall':(100,125)},
    'orange':     {'N':(15,25),  'P':(15,25),  'K':(10,20),  'temp':(10,20), 'humidity':(90,95), 'ph':(6.0,7.5), 'rainfall':(100,150)},
    'papaya':     {'N':(40,60),  'P':(50,70),  'K':(50,70),  'temp':(25,35), 'humidity':(90,95), 'ph':(6.0,7.0), 'rainfall':(100,200)},
    'coconut':    {'N':(15,25),  'P':(15,25),  'K':(30,50),  'temp':(25,35), 'humidity':(80,95), 'ph':(5.0,8.0), 'rainfall':(100,200)},
    'cotton':     {'N':(100,140),'P':(15,25),  'K':(15,25),  'temp':(24,30), 'humidity':(75,80), 'ph':(5.5,8.0), 'rainfall':(60,110)},
    'jute':       {'N':(60,90),  'P':(40,60),  'K':(40,60),  'temp':(24,35), 'humidity':(70,90), 'ph':(6.0,8.0), 'rainfall':(150,250)},
    'coffee':     {'N':(80,120), 'P':(15,25),  'K':(25,40),  'temp':(15,28), 'humidity':(55,65), 'ph':(6.0,7.0), 'rainfall':(150,250)},
}

records = []
for crop, cond in crop_conditions.items():
    for _ in range(100):
        records.append({
            'N':        np.random.randint(*cond['N']),
            'P':        np.random.randint(*cond['P']),
            'K':        np.random.randint(*cond['K']),
            'temperature': round(np.random.uniform(*cond['temp']), 2),
            'humidity':    round(np.random.uniform(*cond['humidity']), 2),
            'ph':          round(np.random.uniform(*cond['ph']), 2),
            'rainfall':    round(np.random.uniform(*cond['rainfall']), 2),
            'label':    crop
        })

df = pd.DataFrame(records).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'✅ Dataset created with {df.shape[0]} rows and {df.shape[1]} columns')
df.head(10)

---
## 🔍 Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
print('=' * 50)
print('📊 DATASET INFORMATION')
print('=' * 50)
print(f'Shape        : {df.shape}')
print(f'Total Rows   : {df.shape[0]}')
print(f'Total Columns: {df.shape[1]}')
print(f'Crops        : {df["label"].nunique()} unique crops')
print()
print('--- Data Types ---')
print(df.dtypes)
print()
print('--- Missing Values ---')
print(df.isnull().sum())

In [ ]:
# Statistical Summary
print('📊 Statistical Summary')
df.describe().round(2)

In [ ]:
# Crop Distribution
plt.figure(figsize=(14, 5))
crop_counts = df['label'].value_counts()
sns.barplot(x=crop_counts.index, y=crop_counts.values, palette='viridis')
plt.title('🌱 Crop Distribution in Dataset', fontsize=16, fontweight='bold')
plt.xlabel('Crop', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
print('✅ Each crop has equal samples — balanced dataset!')

In [ ]:
# Distribution of all features
features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(features):
    axes[i].hist(df[col], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(f'Distribution of {col}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')

axes[-1].axis('off')
plt.suptitle('📊 Feature Distributions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10, 7))
corr = df[features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='coolwarm',
    mask=mask, linewidths=0.5, vmin=-1, vmax=1
)
plt.title('🔗 Feature Correlation Heatmap', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots: Feature vs Crop (Top 6 crops)
top_crops = df['label'].value_counts().head(6).index.tolist()
df_top = df[df['label'].isin(top_crops)]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
plot_features = ['N', 'P', 'K', 'temperature', 'humidity', 'rainfall']

for i, feat in enumerate(plot_features):
    df_top.boxplot(column=feat, by='label', ax=axes[i])
    axes[i].set_title(f'{feat} by Crop', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Crop')
    axes[i].set_ylabel(feat)
    axes[i].tick_params(axis='x', rotation=30)

plt.suptitle('📦 Feature Boxplots by Crop Type', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Average NPK by crop
npk_avg = df.groupby('label')[['N', 'P', 'K']].mean().sort_values('N', ascending=False)

npk_avg.plot(kind='bar', figsize=(16, 6), colormap='Set2', edgecolor='black', width=0.8)
plt.title('🧪 Average NPK Values per Crop', fontsize=15, fontweight='bold')
plt.xlabel('Crop', fontsize=12)
plt.ylabel('Average Value', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.legend(title='Nutrient')
plt.tight_layout()
plt.show()

---
## ⚙️ Step 4: Data Preprocessing

In [ ]:
# Encode Target Label
le = LabelEncoder()
df['crop_encoded'] = le.fit_transform(df['label'])

print('🔢 Crop Label Encoding:')
for i, crop in enumerate(le.classes_):
    print(f'  {i:2d} → {crop}')

In [ ]:
# Features and Target
X = df[features]
y = df['crop_encoded']

# Train-Test Split (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'✅ Training Set  : {X_train.shape[0]} samples')
print(f'✅ Testing Set   : {X_test.shape[0]} samples')
print(f'✅ Features Used : {X_train.shape[1]}')

In [ ]:
# Feature Scaling (needed for Neural Network)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('✅ Feature scaling done using StandardScaler')

---
## 🤖 Step 5: Model Training
### Model 1: Decision Tree Classifier

In [ ]:
dt_model = DecisionTreeClassifier(max_depth=10, random_state=42)
dt_model.fit(X_train, y_train)

dt_pred = dt_model.predict(X_test)
dt_acc  = accuracy_score(y_test, dt_pred)

print('🌳 Decision Tree Classifier')
print(f'   Accuracy : {dt_acc * 100:.2f}%')
print(f'   F1 Score : {f1_score(y_test, dt_pred, average="weighted") * 100:.2f}%')

### Model 2: Random Forest Classifier

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_acc  = accuracy_score(y_test, rf_pred)

print('🌲 Random Forest Classifier')
print(f'   Accuracy : {rf_acc * 100:.2f}%')
print(f'   F1 Score : {f1_score(y_test, rf_pred, average="weighted") * 100:.2f}%')

### Model 3: XGBoost Classifier

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    use_label_encoder=False, eval_metric='mlogloss',
    random_state=42, verbosity=0
)
xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)
xgb_acc  = accuracy_score(y_test, xgb_pred)

print('⚡ XGBoost Classifier')
print(f'   Accuracy : {xgb_acc * 100:.2f}%')
print(f'   F1 Score : {f1_score(y_test, xgb_pred, average="weighted") * 100:.2f}%')

### Model 4: Neural Network (MLP)

In [ ]:
nn_model = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    max_iter=300,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1
)
nn_model.fit(X_train_scaled, y_train)

nn_pred = nn_model.predict(X_test_scaled)
nn_acc  = accuracy_score(y_test, nn_pred)

print('🧠 Neural Network (MLP)')
print(f'   Accuracy : {nn_acc * 100:.2f}%')
print(f'   F1 Score : {f1_score(y_test, nn_pred, average="weighted") * 100:.2f}%')

---
## 📊 Step 6: Model Comparison & Evaluation

In [ ]:
# Model Comparison Table
results = {
    'Model': ['Decision Tree', 'Random Forest', 'XGBoost', 'Neural Network (MLP)'],
    'Accuracy (%)': [
        round(dt_acc * 100, 2),
        round(rf_acc * 100, 2),
        round(xgb_acc * 100, 2),
        round(nn_acc * 100, 2)
    ],
    'F1 Score (%)': [
        round(f1_score(y_test, dt_pred,  average='weighted') * 100, 2),
        round(f1_score(y_test, rf_pred,  average='weighted') * 100, 2),
        round(f1_score(y_test, xgb_pred, average='weighted') * 100, 2),
        round(f1_score(y_test, nn_pred,  average='weighted') * 100, 2)
    ],
    'Precision (%)': [
        round(precision_score(y_test, dt_pred,  average='weighted') * 100, 2),
        round(precision_score(y_test, rf_pred,  average='weighted') * 100, 2),
        round(precision_score(y_test, xgb_pred, average='weighted') * 100, 2),
        round(precision_score(y_test, nn_pred,  average='weighted') * 100, 2)
    ],
    'Recall (%)': [
        round(recall_score(y_test, dt_pred,  average='weighted') * 100, 2),
        round(recall_score(y_test, rf_pred,  average='weighted') * 100, 2),
        round(recall_score(y_test, xgb_pred, average='weighted') * 100, 2),
        round(recall_score(y_test, nn_pred,  average='weighted') * 100, 2)
    ]
}

results_df = pd.DataFrame(results).sort_values('Accuracy (%)', ascending=False).reset_index(drop=True)
print('🏆 Model Comparison Results:')
print(results_df.to_string(index=False))

In [ ]:
# Bar Chart: Model Accuracy Comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']

# Accuracy
axes[0].bar(results_df['Model'], results_df['Accuracy (%)'], color=colors, edgecolor='black', width=0.5)
for i, v in enumerate(results_df['Accuracy (%)']):
    axes[0].text(i, v + 0.3, f'{v}%', ha='center', fontweight='bold')
axes[0].set_title('Model Accuracy Comparison', fontsize=13, fontweight='bold')
axes[0].set_ylim([80, 105])
axes[0].set_ylabel('Accuracy (%)')
axes[0].tick_params(axis='x', rotation=15)

# F1 Score
axes[1].bar(results_df['Model'], results_df['F1 Score (%)'], color=colors, edgecolor='black', width=0.5)
for i, v in enumerate(results_df['F1 Score (%)']):
    axes[1].text(i, v + 0.3, f'{v}%', ha='center', fontweight='bold')
axes[1].set_title('Model F1 Score Comparison', fontsize=13, fontweight='bold')
axes[1].set_ylim([80, 105])
axes[1].set_ylabel('F1 Score (%)')
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('📊 Model Performance Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrix for Best Model (Random Forest)
best_pred = rf_pred
best_name = 'Random Forest'

cm = confusion_matrix(y_test, best_pred)
plt.figure(figsize=(14, 11))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=le.classes_, yticklabels=le.classes_
)
plt.title(f'🔢 Confusion Matrix — {best_name}', fontsize=15, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Classification Report — Random Forest
print(f'📋 Classification Report — {best_name}\n')
print(classification_report(y_test, rf_pred, target_names=le.classes_))

In [ ]:
# Feature Importance — Random Forest
importances = rf_model.feature_importances_
feat_df = pd.DataFrame({'Feature': features, 'Importance': importances})
feat_df = feat_df.sort_values('Importance', ascending=True)

plt.figure(figsize=(9, 5))
colors_imp = plt.cm.viridis(np.linspace(0.2, 0.8, len(feat_df)))
plt.barh(feat_df['Feature'], feat_df['Importance'], color=colors_imp, edgecolor='black')
for i, (feat, imp) in enumerate(zip(feat_df['Feature'], feat_df['Importance'])):
    plt.text(imp + 0.002, i, f'{imp:.3f}', va='center', fontweight='bold')
plt.title('🔍 Feature Importance — Random Forest', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

In [ ]:
# Cross Validation Score
print('📊 Cross Validation Scores (5-Fold):')
for name, model in [('Decision Tree', dt_model), ('Random Forest', rf_model), ('XGBoost', xgb_model)]:
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    print(f'  {name:22s}: Mean={cv_scores.mean()*100:.2f}%  Std={cv_scores.std()*100:.2f}%')

---
## 🎯 Step 7: Crop Prediction Function

In [ ]:
def predict_crop(N, P, K, temperature, humidity, ph, rainfall, model=rf_model, top_n=3):
    """
    Predict the best crop based on soil and weather conditions.

    Parameters:
    -----------
    N, P, K          : Soil nutrients (Nitrogen, Phosphorus, Potassium)
    temperature      : Temperature in Celsius
    humidity         : Humidity in %
    ph               : Soil pH level
    rainfall         : Rainfall in mm
    model            : ML model to use (default: Random Forest)
    top_n            : Number of top alternatives to show

    Returns:
    --------
    Dictionary with prediction and confidence scores
    """
    input_data = np.array([[N, P, K, temperature, humidity, ph, rainfall]])

    # Predict with probabilities
    proba = model.predict_proba(input_data)[0]
    top_indices = np.argsort(proba)[::-1][:top_n]

    recommended_crop = le.classes_[top_indices[0]]
    confidence       = round(proba[top_indices[0]] * 100, 2)
    alternatives     = [
        {'crop': le.classes_[i], 'confidence': round(proba[i] * 100, 2)}
        for i in top_indices[1:]
    ]

    print('=' * 55)
    print('     🌾 CROP RECOMMENDATION RESULT')
    print('=' * 55)
    print(f'  Input Values:')
    print(f'    N={N}, P={P}, K={K}')
    print(f'    Temperature={temperature}°C, Humidity={humidity}%')
    print(f'    pH={ph}, Rainfall={rainfall}mm')
    print('-' * 55)
    print(f'  ✅ Recommended Crop : {recommended_crop.upper()}')
    print(f'  📊 Confidence Score : {confidence}%')
    print()
    print(f'  🔄 Alternative Suggestions:')
    for alt in alternatives:
        print(f'     → {alt["crop"]:15s}  ({alt["confidence"]}% confidence)')
    print('=' * 55)

    return {
        'recommended_crop': recommended_crop,
        'confidence': confidence,
        'alternatives': alternatives
    }

In [ ]:
# --- Test Case 1: Paddy/Rice Field Conditions ---
result1 = predict_crop(N=80, P=50, K=50, temperature=25, humidity=85, ph=6.5, rainfall=250)

In [ ]:
# --- Test Case 2: Dry Conditions ---
result2 = predict_crop(N=20, P=70, K=20, temperature=30, humidity=20, ph=7.5, rainfall=50)

In [ ]:
# --- Test Case 3: Fruit Farming Conditions ---
result3 = predict_crop(N=20, P=20, K=20, temperature=30, humidity=90, ph=6.0, rainfall=150)

In [ ]:
# --- Test Case 4: Custom User Input ---
print('🧑‍🌾 Custom Farmer Input Example')

# Change these values as per your field data
N_val          = 90
P_val          = 42
K_val          = 43
temp_val       = 20
humidity_val   = 82
ph_val         = 6.5
rainfall_val   = 202

result4 = predict_crop(
    N=N_val, P=P_val, K=K_val,
    temperature=temp_val, humidity=humidity_val,
    ph=ph_val, rainfall=rainfall_val
)

---
## 💾 Step 8: Save the Best Model

In [ ]:
# Save the best model and label encoder
joblib.dump(rf_model,  'crop_recommendation_model.pkl')
joblib.dump(le,        'label_encoder.pkl')
joblib.dump(scaler,    'scaler.pkl')

print('💾 Models saved successfully!')
print('   Files created:')
print('   ✅ crop_recommendation_model.pkl')
print('   ✅ label_encoder.pkl')
print('   ✅ scaler.pkl')

---
## 🔄 Step 9: Load & Use the Saved Model

In [ ]:
# Load and re-use saved model
loaded_model   = joblib.load('crop_recommendation_model.pkl')
loaded_encoder = joblib.load('label_encoder.pkl')

def predict_from_saved_model(N, P, K, temperature, humidity, ph, rainfall):
    inp = np.array([[N, P, K, temperature, humidity, ph, rainfall]])
    pred_encoded = loaded_model.predict(inp)[0]
    crop_name    = loaded_encoder.classes_[pred_encoded]
    proba        = loaded_model.predict_proba(inp)[0][pred_encoded]
    print(f'🌾 Recommended Crop : {crop_name.upper()}')
    print(f'📊 Confidence       : {round(proba * 100, 2)}%')

# Test
print('🔁 Using Loaded Saved Model:')
predict_from_saved_model(N=80, P=50, K=50, temperature=25, humidity=85, ph=6.5, rainfall=250)

---
## 📋 Step 10: Final Summary

In [ ]:
print('=' * 60)
print('     ✅ MODULE 1: CROP RECOMMENDATION SYSTEM — SUMMARY')
print('=' * 60)
print()
print('📂 Dataset     : Crop Recommendation Dataset')
print(f'📊 Total Rows  : {df.shape[0]}')
print(f'🌾 Crops       : {df["label"].nunique()} unique crops')
print(f'🔢 Features    : {len(features)} input features')
print()
print('🤖 Model Performance:')
print(f'   Decision Tree  : {dt_acc * 100:.2f}%')
print(f'   Random Forest  : {rf_acc * 100:.2f}%  ⭐ Best Model')
print(f'   XGBoost        : {xgb_acc * 100:.2f}%')
print(f'   Neural Network : {nn_acc * 100:.2f}%')
print()
print('💾 Saved Files :')
print('   crop_recommendation_model.pkl')
print('   label_encoder.pkl')
print('   scaler.pkl')
print('=' * 60)